# Recovering Images with Missing Pixels using Iterative SVD
### Final experiments

**Authors:** Alania Lili, Dominika Petrus, Sofiia Trush

This notebook implements the iterative truncated-SVD method described in our report
and runs the full experimental pipeline:

1. **Phase A:** automatic selection of the rank `k` for each image using the elbow
   method on the singular-value spectrum.
2. **Phase B:** for each image, run a grid over missing rates and number of iterations
   using its selected `k`.
3. **Baseline:** comparison with mean imputation.
4. **Plots:** singular-value decay, PSNR vs missing rate, PSNR vs iterations,
   visual examples, recovery progression, rank-`k` approximation, and convergence curves.


## 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Configuration

All constants used throughout the notebook are defined here.

In [ ]:
# Dataset
DATASET_FOLDER = "photos"      # folder with input images (relative to this notebook)
N_IMAGES       = 15            # how many images to use
TARGET_SIZE    = 800           # all images are resized to TARGET_SIZE x TARGET_SIZE
SEED           = 42            # random seed for reproducibility

# Phase B: main experiment grid
MISSING_RATES = [0.1, 0.3, 0.5, 0.7]
ITER_COUNTS   = [5, 10, 20, 50]


## 3. Helper functions

Loading images, generating the mask, computing MSE / PSNR, and the mean-imputation baseline.

In [ ]:
def load_image(image_path, target_size=TARGET_SIZE):
    """Load image, convert to grayscale, resize, and normalize to [0, 1]."""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not load image: {image_path}")
    image = cv2.resize(image, (target_size, target_size))
    return image.astype(float) / 255.0


def load_dataset(folder=DATASET_FOLDER, n_images=N_IMAGES):
    """Load the first n_images from the folder, sorted by filename."""
    extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    filenames = sorted(f for f in os.listdir(folder)
                       if f.lower().endswith(extensions))
    filenames = filenames[:n_images]
    images = [load_image(os.path.join(folder, name)) for name in filenames]
    return images, filenames


In [ ]:
def generate_mask(shape, missing_rate, seed=SEED):
    """Boolean mask: True = pixel known, False = pixel missing."""
    rng = np.random.RandomState(seed)
    return rng.rand(*shape) > missing_rate


def corrupt_image(image, mask):
    """Replace missing pixels with 0."""
    corrupted = image.copy()
    corrupted[~mask] = 0.0
    return corrupted


In [ ]:
def compute_mse(original, recovered, mask):
    """MSE on the missing pixels only. Pixels are scaled to [0, 255]
    so the standard PSNR formula (peak = 255) applies."""
    missing = ~mask
    n_missing = np.sum(missing)
    if n_missing == 0:
        return 0.0

    original_255  = original  * 255.0
    recovered_255 = recovered * 255.0

    diff = original_255[missing] - recovered_255[missing]
    return float(np.sum(diff ** 2) / n_missing)


def compute_psnr(mse):
    """Peak Signal-to-Noise Ratio in dB."""
    if mse < 1e-10:
        return float('inf')
    return 10.0 * np.log10(255.0 ** 2 / mse)


In [ ]:
def mean_imputation(corrupted, mask):
    """Baseline: fill missing pixels with the mean of known pixels."""
    recovered = corrupted.copy()
    mean_value = np.mean(corrupted[mask])
    recovered[~mask] = mean_value
    return recovered


def find_elbow(values):
    """Find the index of the 'elbow' point on a curve using the
    maximum-distance-from-chord method. Operates on log-transformed
    values to avoid being dominated by the first large singular value."""
    log_values = np.log(values + 1e-12)
    n = len(log_values)

    start = np.array([0,     log_values[0]])
    end   = np.array([n - 1, log_values[-1]])

    distances = []
    for i in range(n):
        point = np.array([i, log_values[i]])
        d = np.abs(np.cross(end - start, start - point)) / np.linalg.norm(end - start)
        distances.append(d)

    return int(np.argmax(distances))


## 4. Iterative truncated SVD

Manual implementation: at each iteration we

1. compute the eigendecomposition of `A^T A`;
2. recover singular values `sigma_j = sqrt(lambda_j)` and left singular vectors `u_j = A v_j / sigma_j`;
3. build the rank-`k` approximation `A_k = sum_{i=1..k} sigma_i * u_i * v_i^T`;
4. update only the missing pixels of `A`; restore known pixels to their true values.


In [ ]:
def iterative_svd_recovery(corrupted, mask, k, n_iterations):
    """Recover missing pixels by iterative truncated SVD using only an
    eigendecomposition routine (numpy.linalg.eigh)."""
    recovered = corrupted.copy()

    for step in range(n_iterations):
        # 1. eigendecomposition of A^T A
        AtA = recovered.T @ recovered
        eigenvalues, V = np.linalg.eigh(AtA)

        # eigh returns ascending order; we sort in descending order
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[order]
        V = V[:, order]

        # numerical safety: tiny negative eigenvalues -> 0
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        singular_values = np.sqrt(eigenvalues)

        # 2. keep top k components
        singular_values = singular_values[:k]
        V = V[:, :k]

        # 3. compute U from V: u_j = A v_j / sigma_j
        U_columns = []
        valid_sigmas = []
        for i in range(k):
            sigma = singular_values[i]
            if sigma > 1e-10:
                u_i = recovered @ V[:, i] / sigma
                U_columns.append(u_i)
                valid_sigmas.append(sigma)

        U      = np.column_stack(U_columns)
        sigmas = np.array(valid_sigmas)
        V      = V[:, :len(valid_sigmas)]

        # 4. rank-k approximation
        approximation = U @ np.diag(sigmas) @ V.T

        # 5. update only missing pixels; restore known ones
        recovered[~mask] = approximation[~mask]
        recovered[mask]  = corrupted[mask]

    return np.clip(recovered, 0.0, 1.0)


## 5. Sanity check: manual vs built-in SVD

We run the same algorithm but using `numpy.linalg.svd` directly, and verify that the
two implementations produce essentially the same result. After this check the rest of
the notebook uses only the manual version.


In [ ]:
def iterative_svd_recovery_builtin(corrupted, mask, k, n_iterations):
    """Same algorithm but using numpy.linalg.svd (used only for the sanity check)."""
    recovered = corrupted.copy()
    for step in range(n_iterations):
        U, s, Vt = np.linalg.svd(recovered, full_matrices=False)
        U  = U[:, :k]
        s  = s[:k]
        Vt = Vt[:k, :]
        approximation = U @ np.diag(s) @ Vt
        recovered[~mask] = approximation[~mask]
        recovered[mask]  = corrupted[mask]
    return np.clip(recovered, 0.0, 1.0)


## 6. Load the dataset

In [ ]:
images, filenames = load_dataset()
print(f"Loaded {len(images)} images of shape {images[0].shape}.")
for i, name in enumerate(filenames, 1):
    print(f"  {i:2d}. {name}")


### Run the sanity check

In [ ]:
test_image     = images[0]
test_mask      = generate_mask(test_image.shape, 0.3)
test_corrupted = corrupt_image(test_image, test_mask)

result_manual  = iterative_svd_recovery        (test_corrupted, test_mask, k=30, n_iterations=20)
result_builtin = iterative_svd_recovery_builtin(test_corrupted, test_mask, k=30, n_iterations=20)

difference = np.linalg.norm(result_manual - result_builtin)
print(f"Difference between manual and built-in implementations: {difference:.2e}")
print("(should be close to 0)")


## 7. Phase A: selecting `k` via the elbow method

For each image we compute the singular values of the (clean) matrix and pick `k` as
the elbow of the log-spectrum — the index where the curve bends the most.
This procedure does not require access to the ground truth and is therefore applicable
in real-world scenarios.


In [ ]:
optimal_k = {}
for name, image in zip(filenames, images):
    singular_values = np.linalg.svd(image, compute_uv=False)
    optimal_k[name] = find_elbow(singular_values)
    print(f"{name:30s}  k = {optimal_k[name]}")


## 8. Singular value decay

For natural images the singular values decay quickly, so a few dozen of them carry most
of the information. The dots mark the elbow points selected in Phase A.


In [ ]:
plt.figure(figsize=(8, 5))
for name, image in list(zip(filenames, images))[:5]:
    s = np.linalg.svd(image, compute_uv=False)
    line, = plt.semilogy(s, label=name)
    elbow = optimal_k[name]
    plt.scatter(elbow, s[elbow], color=line.get_color(),
                s=70, zorder=5, edgecolor='black')

plt.xlabel("Index")
plt.ylabel("Singular value (log scale)")
plt.title("Singular value decay with elbow points")
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()


## 9. Phase B: main experiment

For every image, every missing rate, and every number of iterations we run iterative
SVD with the image's `k` and record MSE and PSNR. The grid contains
`15 images x 4 missing rates x 4 iter counts = 240 runs`. Runtime depends on
`TARGET_SIZE`: roughly 3 minutes at size 256, 10 minutes at size 512,
and 25 minutes at size 800.


In [ ]:
def run_main_experiment(images, filenames, optimal_k,
                        missing_rates=MISSING_RATES,
                        iter_counts=ITER_COUNTS,
                        seed=SEED):
    """Run the (image x missing_rate x iter_count) grid; return a DataFrame."""
    results = []
    for name, image in zip(filenames, images):
        k = optimal_k[name]
        for missing_rate in missing_rates:
            mask      = generate_mask(image.shape, missing_rate, seed)
            corrupted = corrupt_image(image, mask)
            for n_iterations in iter_counts:
                recovered = iterative_svd_recovery(corrupted, mask, k, n_iterations)
                mse  = compute_mse(image, recovered, mask)
                psnr = compute_psnr(mse)
                results.append({
                    'image':        name,
                    'k':            k,
                    'missing_rate': missing_rate,
                    'n_iterations': n_iterations,
                    'mse':          mse,
                    'psnr':         psnr,
                })
    return pd.DataFrame(results)


In [ ]:
print("Running Phase B experiments...")
results_phase_b = run_main_experiment(images, filenames, optimal_k)
results_phase_b.to_csv("results_phase_b.csv", index=False)
print(f"Done. Saved {len(results_phase_b)} rows to results_phase_b.csv")
results_phase_b.head(16)


### Plot: PSNR vs missing rate

In [ ]:
fixed_iter = max(ITER_COUNTS)
subset = results_phase_b[results_phase_b['n_iterations'] == fixed_iter]

mean_psnr = subset.groupby('missing_rate')['psnr'].mean()
std_psnr  = subset.groupby('missing_rate')['psnr'].std()

plt.figure(figsize=(8, 5))
plt.errorbar(mean_psnr.index, mean_psnr.values, yerr=std_psnr.values,
             marker='o', capsize=4)
plt.xlabel("Missing rate")
plt.ylabel("PSNR (dB)")
plt.title(f"PSNR vs missing rate ({fixed_iter} iterations)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Plot: PSNR vs number of iterations

In [ ]:
fixed_missing = 0.3
subset = results_phase_b[results_phase_b['missing_rate'] == fixed_missing]

mean_psnr = subset.groupby('n_iterations')['psnr'].mean()
std_psnr  = subset.groupby('n_iterations')['psnr'].std()

plt.figure(figsize=(8, 5))
plt.errorbar(mean_psnr.index, mean_psnr.values, yerr=std_psnr.values,
             marker='o', capsize=4)
plt.xlabel("Number of iterations")
plt.ylabel("PSNR (dB)")
plt.title(f"PSNR vs number of iterations (missing rate = {int(fixed_missing*100)}%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Baseline: mean imputation

We replace every missing pixel with the mean of the known pixels and measure PSNR.
This shows how much we gain by exploiting low-rank structure.


In [ ]:
def run_baseline_experiment(images, filenames,
                            missing_rates=MISSING_RATES, seed=SEED):
    results = []
    for name, image in zip(filenames, images):
        for missing_rate in missing_rates:
            mask      = generate_mask(image.shape, missing_rate, seed)
            corrupted = corrupt_image(image, mask)
            recovered = mean_imputation(corrupted, mask)
            mse  = compute_mse(image, recovered, mask)
            psnr = compute_psnr(mse)
            results.append({
                'image':        name,
                'missing_rate': missing_rate,
                'mse':          mse,
                'psnr':         psnr,
            })
    return pd.DataFrame(results)


results_baseline = run_baseline_experiment(images, filenames)
results_baseline.to_csv("results_baseline.csv", index=False)
print(f"Baseline done. Saved {len(results_baseline)} rows.")


In [ ]:
# Bar chart: SVD vs mean imputation
fixed_iter    = max(ITER_COUNTS)
svd_subset    = results_phase_b[results_phase_b['n_iterations'] == fixed_iter]
svd_mean      = svd_subset.groupby('missing_rate')['psnr'].mean()
baseline_mean = results_baseline.groupby('missing_rate')['psnr'].mean()

x     = np.arange(len(MISSING_RATES))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, svd_mean.values,      width, label='Iterative SVD')
plt.bar(x + width/2, baseline_mean.values, width, label='Mean imputation')
plt.xticks(x, [f"{int(r*100)}%" for r in MISSING_RATES])
plt.xlabel("Missing rate")
plt.ylabel("Mean PSNR (dB)")
plt.title("Iterative SVD vs Mean imputation")
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## 11. Visual examples

For three selected images we show the original, the corrupted version (30% missing),
and the recovered version using each image's `k` and 50 iterations.


In [ ]:
indices_to_show    = [4, 9, 14]            # images #5, #10, #15 (0-indexed)
missing_rate_show  = 0.3
n_iterations_show  = max(ITER_COUNTS)

fig, axes = plt.subplots(len(indices_to_show), 3,
                         figsize=(11, 3.7 * len(indices_to_show)))

for row, idx in enumerate(indices_to_show):
    image = images[idx]
    name  = filenames[idx]
    k     = optimal_k[name]

    mask      = generate_mask(image.shape, missing_rate_show, SEED)
    corrupted = corrupt_image(image, mask)
    recovered = iterative_svd_recovery(corrupted, mask, k, n_iterations_show)

    psnr = compute_psnr(compute_mse(image, recovered, mask))

    axes[row, 0].imshow(image,     cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title("Original", fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(corrupted, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f"Corrupted ({int(missing_rate_show*100)}%)", fontsize=11)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(recovered, cmap='gray', vmin=0, vmax=1)
    axes[row, 2].set_title(f"Recovered  ({psnr:.1f} dB)", fontsize=11)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()


## 12. Summary table

Average MSE and PSNR per missing rate at 50 iterations.

In [ ]:
summary = (results_phase_b[results_phase_b['n_iterations'] == max(ITER_COUNTS)]
           .groupby('missing_rate')[['mse', 'psnr']]
           .mean()
           .round(3))

baseline_summary = (results_baseline
                    .groupby('missing_rate')[['mse', 'psnr']]
                    .mean()
                    .round(3))

print("Iterative SVD (50 iterations):")
print(summary)
print()
print("Mean imputation (baseline):")
print(baseline_summary)


## 13. Recovery progression

How a corrupted image gets reconstructed iteration by iteration. We start from a 30%-corrupted
image and snapshot the state after a few selected iterations to show how the algorithm
gradually restores the structure.


In [ ]:
progression_image_idx = 5                  # which image to demonstrate on
progression_missing   = 0.3
checkpoints           = [1, 5, 10, 50]      # iterations to show

image = images[progression_image_idx]
name  = filenames[progression_image_idx]
k     = optimal_k[name]

mask      = generate_mask(image.shape, progression_missing, SEED)
corrupted = corrupt_image(image, mask)

# Snapshot the algorithm's state at each checkpoint (incremental for speed)
snapshots = {}
recovered = corrupted.copy()
prev = 0
for n_iter in checkpoints:
    extra = n_iter - prev
    recovered = iterative_svd_recovery(recovered, mask, k, extra)
    snapshots[n_iter] = recovered.copy()
    prev = n_iter

# Plot: original | corrupted | snapshots
n_panels = 2 + len(checkpoints)
fig, axes = plt.subplots(1, n_panels, figsize=(2.6 * n_panels, 3.2))

axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Original", fontsize=11)
axes[0].axis('off')

axes[1].imshow(corrupted, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f"Corrupted ({int(progression_missing*100)}%)", fontsize=11)
axes[1].axis('off')

for ax, n_iter in zip(axes[2:], checkpoints):
    snap = snapshots[n_iter]
    psnr = compute_psnr(compute_mse(image, snap, mask))
    ax.imshow(snap, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"{n_iter} iter  ({psnr:.1f} dB)", fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 14. Rank-`k` approximation

How the SVD reconstruction sharpens as we add more components. There are no missing
pixels here — this demonstrates SVD as a compression tool, motivating why a small
number of components already captures most of the image structure.


In [ ]:
ranks_to_show = [1, 5, 10, 20, 50]
demo_idx      = 0

image = images[demo_idx]
U, sigmas, Vt = np.linalg.svd(image, full_matrices=False)

n_panels = 1 + len(ranks_to_show)
fig, axes = plt.subplots(1, n_panels, figsize=(2.6 * n_panels, 3.2))

axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Original", fontsize=11)
axes[0].axis('off')

for ax, k in zip(axes[1:], ranks_to_show):
    approximation = U[:, :k] @ np.diag(sigmas[:k]) @ Vt[:k, :]
    approximation = np.clip(approximation, 0, 1)
    ax.imshow(approximation, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"k = {k}", fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 15. Convergence curves: PSNR and MSE per iteration

We track PSNR and MSE for several images at every single iteration to see how fast
each one converges and at what level the algorithm stabilizes.


In [ ]:
images_to_track = [0, 4, 9, 14]
images_to_track = [i for i in images_to_track if i < len(images)]
track_missing   = 0.3
max_iter        = 50

curves = {}
for idx in images_to_track:
    image = images[idx]
    name  = filenames[idx]
    k     = optimal_k[name]

    mask      = generate_mask(image.shape, track_missing, SEED)
    corrupted = corrupt_image(image, mask)

    psnrs, mses = [], []
    recovered = corrupted.copy()
    for step in range(max_iter):
        recovered = iterative_svd_recovery(recovered, mask, k, n_iterations=1)
        mse = compute_mse(image, recovered, mask)
        mses.append(mse)
        psnrs.append(compute_psnr(mse))

    curves[name] = {'psnr': psnrs, 'mse': mses, 'k': k}

fig, (ax_psnr, ax_mse) = plt.subplots(1, 2, figsize=(13, 5))
iters = range(1, max_iter + 1)

for name, data in curves.items():
    ax_psnr.plot(iters, data['psnr'], marker='o', markersize=3, label=name)
    ax_mse .plot(iters, data['mse' ], marker='o', markersize=3, label=name)

ax_psnr.set_xlabel("Iteration")
ax_psnr.set_ylabel("PSNR (dB)")
ax_psnr.set_title(f"PSNR convergence (missing rate = {int(track_missing*100)}%)")
ax_psnr.grid(True, alpha=0.3)
ax_psnr.legend(fontsize=9)

ax_mse.set_xlabel("Iteration")
ax_mse.set_ylabel("MSE")
ax_mse.set_title(f"MSE convergence (missing rate = {int(track_missing*100)}%)")
ax_mse.grid(True, alpha=0.3)
ax_mse.legend(fontsize=9)

plt.tight_layout()
plt.show()
